In [37]:
import pandas as pd
import geopandas as gpd
import numpy as np
import pyogrio
import os

In [38]:
folder = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"

### Zonas con IDs estandarizados

In [40]:
zones_standarized = gpd.read_file(os.path.join(folder, "red_shapefiles", "zonas_agebs_standarized.shp"))
zones_standarized = zones_standarized.rename(columns={"id_mun_age":"id_mun_ageb"})

# Map of ageb to visum_id (id_mun_ageb)
ageb_to_visum_id = dict(zip(zones_standarized['clave_ageb'], zones_standarized['id_mun_ageb']))

In [41]:
zones_standarized.dtypes

clave_ageb       object
clave_enti        int64
clave_muni        int64
clave_loca        int64
ageb             object
nombre_mun       object
tipo_ageb        object
poblacion_        int64
area_m2         float64
area_km2        float64
establecim        int64
empleados_      float64
densidad_p      float64
densidad_e      float64
densidad_1      float64
distancia_      float64
consecutiv        int64
id_mun_ageb       int64
geometry       geometry
dtype: object

In [42]:
zones_standarized[zones_standarized['clave_ageb']== '140440011']

,clave_ageb,clave_enti,clave_muni,clave_loca,ageb,nombre_mun,tipo_ageb,poblacion_,area_m2,area_km2,establecim,empleados_,densidad_p,densidad_e,densidad_1,distancia_,consecutiv,id_mun_ageb,geometry
473,140440011,14,44,0,0011,Ixtlahuacán de los Membrillos,rural,15789,9.175450e+07,91.754504,13,2725.438533,172.078746,0.141682,29.703594,32.82126,31,1031,"POLYGON ((2374526.729 941684.611, 2374586.477 ..."


#### Matriz OD de Demanda (que preparo Daniel)

In [43]:
matriz_od_long = pd.read_excel(os.path.join(folder, "Insumos IMEPLAN", "Matriz Origen Destino", "matriz_origen_destino_agebs_HMD_VehiculoPrivado.xlsx"))

# Limpiar string values in Origen and Destino columns
# remove leading and trailing spaces and convert to string type
matriz_od_long['Origen'] = matriz_od_long['Origen'].astype(str).str.strip()
matriz_od_long['Destino'] = matriz_od_long['Destino'].astype(str).str.strip()

# Replace '99999000A' with '14097059A' in Origen and Destino columns
matriz_od_long['Origen'] = matriz_od_long['Origen'].replace('99999000A', '14097059A')
matriz_od_long['Destino'] = matriz_od_long['Destino'].replace('99999000A', '14097059A')

# Map columns Origen & Destino to visum_id for insertion of matrix into visum
matriz_od_long['Origen_map'] = matriz_od_long['Origen'].map(ageb_to_visum_id)
matriz_od_long['Destino_map'] = matriz_od_long['Destino'].map(ageb_to_visum_id)

print(f"Total of {matriz_od_long['Ponderador'].sum():,} trips in the matrix por hora")
matriz_od_long

Total of 345,485 trips in the matrix por hora


,Origen,Destino,Ponderador,Origen_map,Destino_map
0,140440011,140440011,350,1031,1031
1,140440011,140440026,20,1031,1032
2,140440011,1403900012249,27,1031,186
3,140440011,1403900012624,27,1031,216
4,140440011,1404400010098,64,1031,1004
...,...,...,...,...,...
3248,141240001018A,1412400010122,35,9006,9001
3249,141240001018A,1412400010480,70,9006,9030
3250,141240001018A,1412400010495,35,9006,9031
3251,141240001018A,1412400010512,35,9006,9033


In [44]:
# Viajes validos entre las 2,203 zonas
od_zonas_validas = matriz_od_long[
    matriz_od_long['Origen_map'].notna() & matriz_od_long['Destino_map'].notna()
].copy()
od_zonas_validas[['Origen_map', 'Destino_map']] = od_zonas_validas[['Origen_map', 'Destino_map']].astype(int)

# Viajes con accesos carreteros (9999...) que aun no tienen correspondencia de zona
# quedan pendientes para ser asignados a la matriz de visum demanda od
od_zonas_pendientes = matriz_od_long[
    matriz_od_long['Origen_map'].isna() | matriz_od_long['Destino_map'].isna()
].copy()

print(f"Viajes válidos entre las 2,203 zonas: {len(od_zonas_validas):,}")
print(f"Viajes con accesos carreteros que aun no tienen correspondencia de zona: {len(od_zonas_pendientes):,}")

Viajes válidos entre las 2,203 zonas: 3,253
Viajes con accesos carreteros que aun no tienen correspondencia de zona: 0


In [45]:
od_zonas_validas

,Origen,Destino,Ponderador,Origen_map,Destino_map
0,140440011,140440011,350,1031,1031
1,140440011,140440026,20,1031,1032
2,140440011,1403900012249,27,1031,186
3,140440011,1403900012624,27,1031,216
4,140440011,1404400010098,64,1031,1004
...,...,...,...,...,...
3248,141240001018A,1412400010122,35,9006,9001
3249,141240001018A,1412400010480,70,9006,9030
3250,141240001018A,1412400010495,35,9006,9031
3251,141240001018A,1412400010512,35,9006,9033


In [46]:
matriz_od_long.to_excel(os.path.join(folder, "Matriz Origen Destino", "matriz_origen_destino_agebs_tabular_mapeada_prt7a8.xlsx"), index=False)

### Leer Zonas de Visum (2,203)

In [47]:
import win32com.client as com

#Red base GDL (con 2,203 zonas)
red_base = os.path.join(folder, "Red Base GDL", "RedBase 120826", "RedBase 300726 - conectores_final.ver")
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

In [48]:
zone_values = Visum.Net.Zones.GetMultiAttValues("No")

zone_nos = np.array(
    [int(value) for _, value in zone_values],
    dtype=np.int64
)

print(f"Número de zonas en Visum: {len(zone_nos):,}")

Número de zonas en Visum: 2,203


In [ ]:
zone_to_index = {
    zone_no: index
    for index, zone_no in enumerate(zone_nos)
}

# array en ceros
n_zones = len(zone_nos)
demand_array = np.zeros((n_zones, n_zones), dtype=np.float64)

# indices de origen y destino en la matriz
origin_indices = (
    od_zonas_validas
    ["Origen_map"]
    .map(zone_to_index)
    .to_numpy(dtype=np.int64)
)

destination_indices = (
    od_zonas_validas["Destino_map"]
    .map(zone_to_index)
    .to_numpy(dtype=np.int64)
)

# Numero de viajes
demand_values = (
    od_zonas_validas["Ponderador"]
    .to_numpy(dtype=np.float64)
)

# Llenar el array con los viajes reales
np.add.at(
    demand_array,
    (origin_indices, destination_indices),
    demand_values
)

demand_array

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
# Llenar artificialmente con 1 las zonas sin viajes (excepto Tala)
exclude_zones = set(range(4000, 5000))

for zone_no, idx in zone_to_index.items():
    if zone_no in exclude_zones:
        continue

    # Si esa zona no tiene ningún viaje asociado, le metemos 1
    if demand_array[idx, :].sum() == 0 and demand_array[:, idx].sum() == 0:
        demand_array[idx, idx] = 1

In [50]:
print("Dimensión:", demand_array.shape)
print(f"Demanda total CSV: {od_zonas_validas['Ponderador'].sum():,.3f}")
print(f"Demanda total matriz: {demand_array.sum():,.3f}")
print(f"Pares con demanda: {np.count_nonzero(demand_array):,}")

Dimensión: (2203, 2203)
Demanda total CSV: 345,485.000
Demanda total matriz: 345,485.000
Pares con demanda: 3,253


### Get Visum matrix to fill

In [51]:
matrix_no = 3

visum_matrix = Visum.Net.Matrices.ItemByKey(matrix_no)
print("Matrix No:", visum_matrix.AttValue("No"))
print("Name:", visum_matrix.AttValue("Name"))

Matrix No: 3.0
Name: Auto OD 7a8


In [52]:
visum_matrix.SetValues(
    demand_array,
    Add=False
)